# 1. Imports

In [1]:
import pandas as pd
import numpy as np

# 2. Load the train data file and label file

In [29]:
df_train = pd.read_csv('../data/merged_ble_data.csv')
df_labels = pd.read_csv('../data/location_train_labels.csv')

In [30]:
df_train.columns

Index(['user_id', 'timestamp', 'mac_address', 'RSSI', 'power',
       'year_month_day', 'hour', 'RSSI_1', 'RSSI_2', 'RSSI_3', 'RSSI_4',
       'RSSI_5', 'RSSI_6', 'RSSI_7', 'RSSI_8', 'RSSI_9', 'RSSI_10', 'RSSI_11',
       'RSSI_12', 'RSSI_13', 'RSSI_14', 'RSSI_15', 'RSSI_16', 'RSSI_17',
       'RSSI_18', 'RSSI_19', 'RSSI_20', 'RSSI_21', 'RSSI_22', 'RSSI_23',
       'RSSI_24', 'RSSI_25'],
      dtype='object')

In [31]:
df_labels.columns

Index(['Unnamed: 0', 'Unnamed: 0.1', 'started_at', 'finished_at', 'user_id',
       'user', 'room', 'floor', 'duration', 'duration_min'],
      dtype='object')

# 3. Label the train data

In [32]:
df_train = df_train.drop(columns=['started_at', 'finished_at', 'user', 'room', 'floor'], errors='ignore')

df_train['timestamp'] = pd.to_datetime(df_train['timestamp']).dt.tz_localize(None)
df_labels['started_at'] = pd.to_datetime(df_labels['started_at']).dt.tz_localize(None)
df_labels['finished_at'] = pd.to_datetime(df_labels['finished_at']).dt.tz_localize(None)

df_train = df_train.sort_values('timestamp')
df_labels = df_labels.sort_values('started_at')

df_merged = pd.merge_asof(
    df_train, 
    df_labels[['started_at', 'finished_at', 'user', 'room', 'floor']], 
    left_on='timestamp', 
    right_on='started_at',
    direction='backward'
)

print(f"Total rows: {len(df_merged)}")
print(f"Matched rows before time-window check: {df_merged['user'].notna().sum()}")

mask = df_merged['timestamp'] > df_merged['finished_at']
df_merged.loc[mask, ['user', 'room', 'floor']] = None

df_merged = df_merged.drop(columns=['started_at', 'finished_at'])

df_merged.head()

Total rows: 5005751
Matched rows before time-window check: 4320928


,user_id,timestamp,mac_address,RSSI,power,year_month_day,hour,RSSI_1,RSSI_2,RSSI_3,...,RSSI_19,RSSI_20,RSSI_21,RSSI_22,RSSI_23,RSSI_24,RSSI_25,user,room,floor
0,90,2023-04-10 10:22:55.589,FD:07:0E:D5:28:AE,-75.0,-2147483648,2023-04-10,10,0,0,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
1,90,2023-04-10 10:22:55.595,FD:07:0E:D5:28:AE,-75.0,-2147483648,2023-04-10,10,0,0,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
2,90,2023-04-10 10:22:55.601,FD:07:0E:D5:28:AE,-75.0,-2147483648,2023-04-10,10,0,0,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
3,90,2023-04-10 10:22:55.605,FD:07:0E:D5:28:AE,-75.0,-2147483648,2023-04-10,10,0,0,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
4,90,2023-04-10 10:22:55.609,D2:1C:25:72:FB:E3,-62.0,-2147483648,2023-04-10,10,0,0,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN


In [33]:
# drop the rows where user, room, or floor is NaN
print(f"Total rows before labeling: {len(df_merged)}")
df_train_labeled = df_merged.dropna(subset=['user', 'room', 'floor'])
print(f"Total rows after labeling: {len(df_train_labeled)}")

Total rows before labeling: 5005751
Total rows after labeling: 2573868


In [34]:
df_train_labeled = df_train_labeled.drop(columns=['year_month_day', 'hour', 'user', 'floor'])
df_train_labeled = df_train_labeled.drop(columns=df_train_labeled.columns[5:30])
df_train_labeled.head()

,user_id,timestamp,mac_address,RSSI,power,room
684823,90,2023-04-10 14:21:46.003,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen
684824,90,2023-04-10 14:21:46.008,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen
684825,90,2023-04-10 14:21:46.012,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen
684826,90,2023-04-10 14:21:46.018,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen
684827,90,2023-04-10 14:21:46.024,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen


In [39]:
df_train_labeled = df_train_labeled.rename(columns={'room':'location'})
df_train_labeled['location'].unique()

array(['kitchen', 'cafeteria', 'cleaning', 'nurse station', 'hallway',
       '501', '506', '520', '511', '523', '516', '508', 'Office Large',
       'Office Small', '202', 'Nurse Room', 'Cafeteria C', '213',
       'Cafeteria A', '206', '212', '211', '210', '208', '207',
       'Cafeteria D', '512', 'Cafeteria B', '522', 'Clean Room', '502',
       '518', '513', '201', 'Bathroom', 'Clean 9', '203', '517', '515',
       '503', '205', '505'], dtype=object)

In [41]:
rooms_to_remove = ['201', '202', '203', '206', '207', '208', '210', '211', '212', '213', 'Office Large', 'Office Small']
df_train_labeled = df_train_labeled[~df_train_labeled['location'].isin(rooms_to_remove)]
df_train_labeled['location'] = df_train_labeled['location'].replace({'Nurse Room': 'nurse station', 
                                       'Cafeteria A': 'cafeteria', 
                                       'Cafeteria B': 'cafeteria', 
                                       'Cafeteria C': 'cafeteria',
                                       'Cafeteria D': 'cafeteria',
                                       'Clean Room': 'cleaning',
                                       'Clean 9': 'cleaning'})

In [ ]:
labels = df_train_labeled['location'].unique()
print(f"Number of labels: {len(labels)}")

23
